In [0]:
%sql
WITH ultima_fecha AS (
    SELECT
        MAX(DATE(fecha_medicion)) AS fecha_ultimo_dato
    FROM quality_data.gold.alertas_calidad
),

resumen_hojas_ruta AS (
    SELECT
        a.hdr,
        a.cliente_codigo,
        a.articulo_codigo,
        COUNT(*) AS total_alertas,
        COUNT(DISTINCT a.variable_nombre) AS total_variables_fuera_limite,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(a.variable_nombre))) AS variables_fuera_limite,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(a.estado_control))) AS estados_detectados,
        ROUND(MAX(a.distancia_al_limite), 2) AS mayor_distancia_al_limite,
        MAX(a.fecha_medicion) AS ultima_fecha_alerta,
        CASE
            WHEN COUNT(DISTINCT a.variable_nombre) >= 3 OR COUNT(*) >= 5 THEN 'ALTA'
            WHEN COUNT(DISTINCT a.variable_nombre) = 2 OR COUNT(*) >= 3 THEN 'MEDIA'
            ELSE 'BAJA'
        END AS prioridad
    FROM quality_data.gold.alertas_calidad a
    CROSS JOIN ultima_fecha u
    WHERE DATE(a.fecha_medicion) = u.fecha_ultimo_dato
    GROUP BY
        a.hdr,
        a.cliente_codigo,
        a.articulo_codigo
)

SELECT
    hdr,
    cliente_codigo,
    articulo_codigo,
    total_alertas,
    total_variables_fuera_limite,
    variables_fuera_limite,
    estados_detectados,
    mayor_distancia_al_limite,
    prioridad,
    ultima_fecha_alerta
FROM resumen_hojas_ruta
ORDER BY
    CASE prioridad
        WHEN 'ALTA' THEN 1
        WHEN 'MEDIA' THEN 2
        ELSE 3
    END,
    total_alertas DESC,
    total_variables_fuera_limite DESC,
    mayor_distancia_al_limite DESC
LIMIT 100;